## Extract vital signs and nursing reports for the first ICU stay

### Prerequisite: build the following derived tables according to official repo https://github.com/MIT-LCP/mimic-code/tree/main/mimic-iii
echo_data;
height_first_day;
pivoted_vital;
weight_durations;
pivoted_fio2;
pivoted_gcs;
pivoted_lab;

### the tables below can be built using our code
first_stay_survive;
capillary;
ph;


In [1]:
#libraries
import numpy as np
import pandas as pd
import psycopg2 #used to connect to our local MIMIC-III database
import collections
# import getpass
from datetime import datetime
import os,sys,re
import pickle
import csv
import math
#import seaborn as sns
# import random
from datetime import timedelta
from pathlib import Path
import importlib
import bisect
import glob
import statistics


from notebook.services.config import ConfigManager
cm = ConfigManager()
cm.update('livereveal', {
        'width': 1024,
        'height': 768,
        'scroll': True,
})

#%load_ext autotime

{'width': 1024, 'height': 768, 'scroll': True}

In [2]:
## try to connect MIMICIII

dbname = 'mimiciii'
password = 'YourPassword'
user = 'postgres'
conn = psycopg2.connect(dbname=dbname, password=password,user=user,port=5433)
cur=conn.cursor()

In [1]:
## for documentation to see the note time of the last nursing report and icu outtime
'''
select icu.icustay_id, icu.intime, icu.outtime, t1.last_notetime from (select icu.icustay_id,  max(note.charttime) 
last_notetime from icustays icu join noteevents note on icu.subject_id = note.subject_id
 where note.charttime <= icu.outtime and note.charttime >= icu.intime and note.category = 'Nursing/other' and 
 note.description = 'Report' and note.cgid in (select cgid from public.caregivers where label = 'RN') and 
 length(note.text)>100 group by icu.icustay_id) t1 join icustays icu on icu.icustay_id = t1.icustay_id limit 20


'''

### Capillary refill and pH

In [ ]:
'''
in chartevents for capillary refill
itemid = 3348, value = 'Brisk' or 'Delayed'
itemid = 223951 (capillary right) or 224308 (capillary left), value = 'Normal <3 Seconds' or 'Abnormal >3 Seconds'

'''

# build capillary refill table if not exsists

query = '''
create table capillary as 
select subject_id, icustay_id, charttime, case when
(value like 'Brisk' or value like 'Normal <3 Seconds') 
then 1 else 0 end as capillary_normal,
case when (value like 'Delayed' or value like 'Abnormal >3 Seconds')
then 1 else 0 end as capillary_abnormal 
from chartevents where itemid = 3348 or itemid = 223951

'''
cur.execute(query)


# build ph table if not exsists


query = '''

create table pH as

select lab.subject_id, icu.icustay_id, lab.charttime, lab.valuenum as pH from labevents lab join icustays icu on
lab.subject_id = icu.subject_id where lab.itemid = 50820 and lab.valuenum is not null and icu.intime <= lab.charttime and
icu.outtime >= lab.charttime
'''
cur.execute(query)

In [3]:
from nltk import sent_tokenize, word_tokenize
import nltk
#nltk.download('punkt_tab') #download vocab if necessarr
#nltk.download('punkt') 
import re
import torch


SECTION_TITLES = re.compile(
    r'(NEURO|CV|GU/GI|GU|GI|RESP|ENDO|PLAN|SOCIAL|CARDIAC|ENDO|CVS|LYTES|SKIN|O|P'
    r'|G&D|FEN|F&N|DEV|PARENTS|BILI|SEPSIS|DEVE|PARENTING|FEN O|G&D O|INC|ACTIVITY'
    r'|ID|[** **]|#1|#2|#3|#4|#5|#6|#7|#8|#9|#10):|#1|#2|#3|#4|#5|#6|#7|#8'
    r'|#9|#10|1.\)|2.\)|3.\)|4.\)|5.\)|6.\)|7.\)|8.\)|9.\)|10.\)',re.I | re.M)


max_length = 128


def split_heading(text):
    """Split the report into sections"""
    start = 0
    for matcher in SECTION_TITLES.finditer(text):
        # add last
        end = matcher.start()
        if end != start:
            section = text[start:end].strip()
            if section:
                yield section

        # add title
        start = end
        end = matcher.end()
        if end != start:
            section = text[start:end].strip()
            if section:
                yield section

        start = end

    # add last piece
    end = len(text)
    if start < end:
        section = text[start:end].strip()
        if section:
            yield section


            


def clean_text(text):
    """
    Clean text
    """

    # Replace [**Patterns**] with spaces.
    text = re.sub(r'\[\*\*.*?\*\*\]', pattern_repl, text)
    # Replace `_` with spaces.
    text = re.sub(r'_', ' ', text)

    start = 0
    new_text = ''
    if start > 0:
        new_text += ' ' * start
    new_text = text[start:]
    end = len(text)

    # make sure the new text has the same length of old text.
    if len(text) - end > 0:
        new_text += ' ' * (len(text) - end)
    return new_text


def preprocess_mimic(text):
    """
    Preprocess reports in MIMIC-III.
    1. remove [**Patterns**] and signature
    2. split the report into sections
    3. tokenize sentences and words
    4. lowercase
    """
    for sec in split_heading(clean_text(text)):
        for sent in sent_tokenize(sec):
            text = ' '.join(word_tokenize(sent))
            yield text.lower()
            

def getSentences(t):
    return list(preprocess_mimic(t))




def pattern_repl(matchobj):
    """
    Return a replacement string to be used for match object
    """
    return ' '.rjust(len(matchobj.group(0)))




def txt2embd(text, tokenizer, bert):
    # text: str
    
    # split report into sentences str ----> [str1, str2,...]
    sentences = getSentences(text)
    if len(sentences) == 0:
        return None
    encoded_sent = tokenizer.encode_plus(
                    text=sentences,                      # Preprocess sentence
                    add_special_tokens=True,        # Add [CLS] and [SEP]
                    max_length=max_length,             # Max length to truncate/pad
                    pad_to_max_length=True,         # Pad sentence to max length
                    #return_tensors='pt',           # Return PyTorch tensor
                    return_attention_mask=True,     # Return attention mask
                    truncation=True
                    )
    Textarr = torch.tensor([encoded_sent.get('input_ids')])
    Attnarr = torch.tensor([encoded_sent.get('attention_mask')])
    txtemb = bert.bert(Textarr, Attnarr)
    emb = txtemb[0][0] # (128, 768)
    # Attnarr[0].tolist() (128)
    print(Attnarr[0].tolist())
    return emb.tolist(), Attnarr[0].tolist()
    
def load_BERT(freeze=True):
    # freeze: load BERT without backpropagation
    path = '../clinicalbert_cache/ClinicalBERT.pickle'
    clinical_bert = Base_ClinicalBERT(path, freeze=freeze)
    path = '../clinicalbert_cache/tokenizer.pickle'
    tokenizer = load_tokenizer(path)
    return clinical_bert, tokenizer


class Base_ClinicalBERT(torch.nn.Module):
    def __init__(self, path, freeze=False):
        super(Base_ClinicalBERT, self).__init__()
        self.model_name = 'emilyalsentzer/Bio_ClinicalBERT'
        if os.path.isfile(path):
            with open(path, 'rb') as file:
                self.bert = pickle.load(file) 
        else:
            from transformers import BertModel
            self.bert = BertModel.from_pretrained(self.model_name,
                                                  return_dict=False)
            with open(path, 'wb') as file:
                pickle.dump(self.bert, file)
        if freeze:
            for p in self.bert.parameters():
                p.requires_grad = False
            
def load_tokenizer(path):
    if os.path.isfile(path):
        with open(path, 'rb') as file:
            tokenizer = pickle.load(file)
    else:
        from transformers import BertTokenizer
        tokenizer = BertTokenizer.from_pretrained('emilyalsentzer/Bio_ClinicalBERT',
                                                  do_lower_case=True)
        with open(path, 'wb') as file:
            pickle.dump(tokenizer, file)
    return tokenizer



def preprocess_clinicalbert(text):
    text = text.replace('\n', ' ')
    text = text.replace('\r', ' ')
    text = text.strip().lower()
    text = re.sub('\\[(.*?)\\]', '', text)  # remove de-identified brackets
    text = re.sub('[0-9]+\.', '', text)  # remove 1.2. since the segmenter segments based on this
    text = re.sub('m\.d\.', 'md', text)
    text = re.sub('admission date:', '', text)
    text = re.sub('discharge date:', '', text)
    text = re.sub('--|__|==', '', text)
    return text


def get_preprocessed_text(notes):
    # notes: {id:{tim1:note1,...}}
    exclude = []
    for stay_id in notes:
        idv = notes[stay_id]
        exclude_time = []
        times = list(notes[stay_id].keys())
        for time in times:
            sentences = preprocess_clinicalbert(idv[time][0])
            if len(sentences) == 0:
                exclude_time.append(time)
            else:
                notes[stay_id][time] = sentences
        if len(exclude_time) > 0:
            del notes[stay_id][time]
        if len(notes[stay_id].keys()) == 0:
            exclude.append(stay_id)
    if len(exclude) > 0:
        for stay_id in exclude:
            del notes[stay_id]
    return notes





def get_split_sentences(notes):
    # notes: {id:{tim1:note1,...}}
    exclude = []
    for stay_id in notes:
        idv = notes[stay_id]
        exclude_time = []
        times = list(notes[stay_id].keys())
        for time in times:
            sentences = getSentences(idv[time][0])
            if len(sentences) == 0:
                exclude_time.append(time)
            else:
                notes[stay_id][time] = sentences
        if len(exclude_time) > 0:
            del notes[stay_id][time]
        if len(notes[stay_id].keys()) == 0:
            exclude.append(stay_id)
    if len(exclude) > 0:
        for stay_id in exclude:
            del notes[stay_id]
    return notes


def get_emb(notes):
    # notes: {id:{tim1:note1,...}}
    bert, tokenizer = load_BERT()
    exclude = []
    for stay_id in notes:
        times = list(notes[stay_id].keys())
        idv = notes[stay_id]
        exclude_time = []
        for time in times:
            emb_attn = txt2embd(idv[time][0], tokenizer, bert)
            if emb_attn is None:
                exclude_time.append(time)
            else:
                notes[stay_id][time] = emb_attn # (emb, attn_mask)
        if len(exclude_time) > 0:
            for time in exclude_time:
                del notes[stay_id][time]
        if len(notes[stay_id].keys()) == 0:
            exclude.append(stay_id)
    if len(exclude) > 0:
        for stay_id in exclude:
            del notes[stay_id]
    return notes








def median_impute(df, columns):
    # df has column icustay_id
    for col in columns:
        median = df[col].dropna(how='any').median() # global median
        stay_median = {}
        for stay,val in zip(df['icustay_id'], df[col]):
            if stay not in stay_median:
                stay_median[stay] = []
            if not math.isnan(float(val)):
                stay_median[stay].append(float(val))
        stay_median1 = {}
        for i in stay_median:
            vals = stay_median[i]
            if len(vals) == 0:
                stay_median1[i] = median # global median
            else:
                stay_median1[i] = np.median(vals) # local median
        new_values = []
        for stay,val in zip(df['icustay_id'], df[col]):
            if math.isnan(float(val)):
                new_values.append(stay_median1[stay])
            else:
                new_values.append(float(val))
        df[col] = new_values
    return df

def datetime_to_sec(df,time_col='charttime'):
    time_in_sec = []
    date_format = '%Y-%m-%d %H:%M:%S'
    for i in df[time_col]:
        time_in_sec.append(datetime.strptime(str(i),date_format).timestamp())
    df[time_col] = time_in_sec
    return df


def feature_dict(df, feature_start=2, stay=0, time=1):
    result = {}
    n = len(df)
    for i in range(n):
        row = list(df.iloc[i])
        stay_id = row[stay]
        charttime = row[time]
        features = row[feature_start:]
        if stay_id not in result:
            result[stay_id] = {}
        result[stay_id][charttime] = features
    return result # {icustay_id:{charttime1:[features],charttime2:[features],...}, ...} 


def label_dict(df, label_start=1, stay=0):
    result = {}
    n = len(df)
    for i in range(n):
        row = list(df.iloc[i])
        stay_id = row[stay]
        features = row[label_start:]
        result[stay_id] = features
    return result



def down_sample(dict1, interval=1800):
    # interval: sec
    # down sample the df if there too many records
    result = {}
    for stay_id in dict1:
        result[stay_id] = {}
        times = sorted(list(dict1[stay_id].keys()))
        current = times[0]
        total = len(times)
        last_idx = -1
        while bisect.bisect_left(times, current) < total:
            idx = bisect.bisect_left(times, current)
            current += interval
            if last_idx == idx:
                continue
            else:
                last_idx = idx
            timestamp = times[idx]
            result[stay_id][timestamp] = dict1[stay_id][timestamp]
    return result


def get_median(data_dict):
    data = []
    median = []
    for stay_id in data_dict:
        for time in data_dict[stay_id].keys():
            data.append(data_dict[stay_id][time])
    col = len(data[0])
    data = np.array(data)
    for i in range(col):
        median.append(statistics.median(data[:,i]))
    return median
    

def merge_dict(df1, df2):
    # df1: dict, df2: dict
    # iterate timestamps of df1
    result = {}
    global_median = get_median(df2)
    for stay_id in df1:
        if stay_id not in df2:
            df2[stay_id] = {}
            starttime = min(list(df1[stay_id].keys()))
            df2[stay_id][starttime] = global_median
        result[stay_id] = {}
        times1 = sorted(list(df1[stay_id].keys()))
        times2 = sorted(list(df2[stay_id].keys()))
        total = len(times2)
        for timestamp in times1:
            idx = bisect.bisect_left(times2, timestamp)
            if idx == total:
                timestamp2 = times2[-1]
            else:
                timestamp2 = times2[idx]
            result[stay_id][timestamp] = df1[stay_id][timestamp]
            result[stay_id][timestamp].extend(df2[stay_id][timestamp2])
    del df2
    return result
    

## Extarct cohort: patients survived from first stay

In [3]:
query = '''
DROP TABLE IF EXISTS first_stay_survive;
create table first_stay_survive as (
select stay.subject_id, stay.icustay_id,stay.intime, stay.outtime  from (
select rk.subject_id, rk.icustay_id, rk.intime,rk.outtime, rk.los from ( SELECT subject_id, icustay_id, intime,outtime, los, 
RANK() OVER (PARTITION BY subject_id ORDER BY intime asc) as RN
FROM icustays) rk where rk.rn=1) stay join patients pt on stay.subject_id = pt.subject_id where pt.dod_hosp > stay.outtime
or pt.dod_hosp is null)
'''

cur.execute(query)

## Extract height

In [4]:
query = '''

select icustay_id, height from height_first_day where icustay_id in (select icustay_id from first_stay_survive) and 
height is not null
''' 
height = label_dict(pd.read_sql_query(query,conn))

## Extract dynamic features

In [5]:
path = './dynamic_features.pickle'
if os.path.isfile(path):
    file = open(path,'rb')
    dynamic_features = pickle.load(file)
    file.close()
else:
    
    query ='''
select icustay_id, charttime, heartrate, sysbp, diasbp, meanbp, resprate, tempc, spo2, glucose from pivoted_vital where
icustay_id in (select icustay_id from first_stay_survive)

'''
    pivoted_vital = pd.read_sql_query(query,conn)
    pivoted_vital = median_impute(pivoted_vital, ['heartrate', 'sysbp', 'diasbp', 'meanbp', 'resprate', 'tempc', 'spo2', 
                                                  'glucose'])
    pivoted_vital = down_sample(feature_dict(datetime_to_sec(pivoted_vital)))
    dynamic_features = pivoted_vital
    del pivoted_vital
    
    
    query = '''
    
    select icustay_id, charttime, aniongap, albumin, bands, bicarbonate, bilirubin, creatinine, chloride, glucose, hematocrit,
    hemoglobin, lactate, platelet, potassium, ptt, inr, pt, sodium, bun, wbc 
    from pivoted_lab where icustay_id in (select icustay_id from first_stay_survive)
    '''
    pivoted_lab = pd.read_sql_query(query,conn)
    pivoted_lab = feature_dict(datetime_to_sec(median_impute(pivoted_lab, ['aniongap', 'albumin', 'bands', 'bicarbonate', 'bilirubin', 
                                             'creatinine', 'chloride', 'glucose', 'hematocrit', 'hemoglobin',
                                             'lactate', 'platelet', 'potassium', 'ptt', 'inr',
                                             'pt', 'sodium', 'bun', 'wbc'])))
    dynamic_features = merge_dict(dynamic_features, pivoted_lab)
    
    
    query ='''
select icustay_id, charttime, capillary_normal, capillary_abnormal from capillary where
icustay_id in (select icustay_id from first_stay_survive) and capillary_normal is not null and capillary_abnormal is not null
'''
    capillary = pd.read_sql_query(query,conn)
    capillary = feature_dict(datetime_to_sec(capillary))
    dynamic_features = merge_dict(dynamic_features, capillary)
    
    query = '''
    
    select icustay_id, starttime as charttime, weight from weight_durations where icustay_id in (select icustay_id 
    from first_stay_survive)
    '''
    weight = pd.read_sql_query(query,conn)
    weight = feature_dict(datetime_to_sec(weight))
    dynamic_features = merge_dict(dynamic_features, weight)
    
    
    query ='''
select icustay_id, charttime, fio2 from pivoted_fio2 where
icustay_id in (select icustay_id from first_stay_survive) and fio2 is not null

'''
    pivoted_fio2 = pd.read_sql_query(query,conn)
    pivoted_fio2 = feature_dict(datetime_to_sec(pivoted_fio2))
    dynamic_features = merge_dict(dynamic_features, pivoted_fio2)
    
    query ='''
select icustay_id, charttime, gcs, gcsmotor, gcsverbal, gcseyes from pivoted_gcs where
icustay_id in (select icustay_id from first_stay_survive)

'''
    gcs = pd.read_sql_query(query,conn)
    gcs = median_impute(gcs, ['gcs', 'gcsmotor', 'gcsverbal', 'gcseyes'])
    gcs = feature_dict(datetime_to_sec(gcs))
    dynamic_features = merge_dict(dynamic_features, gcs)
    
    query ='''
select icustay_id, charttime, ph from ph where
icustay_id in (select icustay_id from first_stay_survive) and ph is not null

'''
    ph = pd.read_sql_query(query,conn)
    ph = feature_dict(datetime_to_sec(ph))
    dynamic_features = merge_dict(dynamic_features, ph)
    
    
    
    #### exclude some high missingness patients
    query = '''
    select distinct icustay_id from pivoted_vital where icustay_id in (select icustay_id from first_stay_survive)
    '''
    pivoted_vital_id = list(pd.read_sql_query(query,conn)['icustay_id'])
    
    
    query = '''
    select distinct icustay_id from pivoted_lab where icustay_id in (select icustay_id from first_stay_sepsis)
    '''
    pivoted_lab_id = list(pd.read_sql_query(query,conn)['icustay_id'])
    
    query = '''
    select distinct icustay_id from capillary where icustay_id in (select icustay_id from first_stay_survive)
    '''
    capillary_id = list(pd.read_sql_query(query,conn)['icustay_id'])
    query = '''
    select distinct icustay_id from weight_durations where icustay_id in (select icustay_id from first_stay_survive)
    '''
    weight_id = list(pd.read_sql_query(query,conn)['icustay_id'])
    query = '''
    select distinct icustay_id from pivoted_fio2 where icustay_id in (select icustay_id from first_stay_survive)
    '''
    pivoted_fio2_id = list(pd.read_sql_query(query,conn)['icustay_id'])
    query = '''
    select distinct icustay_id from pivoted_gcs where icustay_id in (select icustay_id from first_stay_survive)
    '''
    pivoted_gcs_id = list(pd.read_sql_query(query,conn)['icustay_id'])
    query = '''
    select distinct icustay_id from ph where icustay_id in (select icustay_id from first_stay_survive)
    '''
    ph_id = list(pd.read_sql_query(query,conn)['icustay_id'])
    
    exclude_id = []
    min_num_missing = 3 # minimum acceptable
    for icustay_id in pivoted_vital_id:
        missing = 0
        if icustay_id not in pivoted_lab_id:
            missing += 1
        if icustay_id not in capillary_id:
            missing += 1
        if icustay_id not in weight_id:
            missing += 1
        if icustay_id not in pivoted_fio2_id:
            missing += 1
        if icustay_id not in pivoted_gcs_id:
            missing += 1
        if icustay_id not in ph_id:
            missing += 1
        if missing > min_num_missing:
            exclude_id.append(icustay_id)
            
    for icustay_id in exclude_id:
        del dynamic_features[icustay_id]
    file = open(path,'wb')
    pickle.dump(dynamic_features,file)
    file.close()




## Merge dynamic features and height

In [6]:
temp = []
for stay_id in height:
    temp.append(height[stay_id][0])
height_median = statistics.median(temp)
for stay_id in dynamic_features:
    if stay_id in height:
        height_value = height[stay_id][0]
    else:
        height_value = height_median
    for timestamp in dynamic_features[stay_id]:
        dynamic_features[stay_id][timestamp].append(height_value)

## Extract nursing reports

In [7]:
path = './notes.pickle'
if os.path.isfile(path):
    file = open(path,'rb')
    notes = pickle.load(file)
    file.close()
else:
    query = '''

select first.icustay_id, note.charttime, note.text from noteevents note join first_stay first on 
first.subject_id = note.subject_id where note.charttime >= first.intime and note.charttime <= first.outtime and
note.category = 'Nursing/other' and note.description = 'Report' and 
note.cgid in (select cgid from public.caregivers where label = 'RN') and length(note.text)>100 and 
note.text not like '%Addendum%' and note.text not like '%addendum%' and note.text not like '%ADDENDUM%' 
and note.text not like '%Management Note%' and note.text not like '%management note%' and note.text not like '%Management note%'
and note.text not like '%MANAGEMENT NOTE%' and note.text not like '%Admission Note%' and note.text not like '%Admission note%'
and note.text not like '%admission note%' and note.text not like '%ADMISSION NOTE%'
'''

    #notes = get_split_sentences(feature_dict(datetime_to_sec(pd.read_sql_query(query,conn))))
    notes = get_preprocessed_text(feature_dict(datetime_to_sec(pd.read_sql_query(query,conn))))
    file = open(path,'wb')
    pickle.dump(notes,file)
    file.close()

## Extract labels

In [8]:
### select patients admitted to ICU more than once
#query = '''
#select first.icustay_id, case when (second.intime is null or extract(day from second.intime - first.outtime)>30) 
#then 0 else 1 end as readmission from first_stay_survive first LEFT JOIN (

#select rk.subject_id, rk.icustay_id, rk.intime from ( SELECT subject_id, icustay_id, intime,outtime, 
#RANK() OVER (PARTITION BY subject_id ORDER BY intime asc) as RN
#FROM icustays) rk where rk.rn=2) second on first.subject_id = second.subject_id
#'''

### similar to above, but only patient with length of stay less than XX hours, excluding those died in the first stay
#query = '''
#select * from (
#select first.icustay_id, case when (second.intime is null or extract(day from second.intime - first.outtime)>30) 
#then 0 else 1 end as readmission from first_stay_survive first LEFT JOIN (
#
#select rk.subject_id, rk.icustay_id, rk.intime from ( SELECT subject_id, icustay_id, intime,outtime, 
#RANK() OVER (PARTITION BY subject_id ORDER BY intime asc) as RN
#FROM icustays) rk where rk.rn=2) second on first.subject_id = second.subject_id
#) t1 where t1.icustay_id in (select icustay_id from first_stay where los <=72)
#'''

#query = '''
#select * from (
#select first.icustay_id, case when (second.intime is null or extract(day from second.intime - first.outtime)>30) 
#then 0 else 1 end as readmission from first_stay first LEFT JOIN (
#
#select rk.subject_id, rk.icustay_id, rk.intime from ( SELECT subject_id, icustay_id, intime,outtime, 
#RANK() OVER (PARTITION BY subject_id ORDER BY intime asc) as RN
#FROM icustays) rk where rk.rn=2) second on first.subject_id = second.subject_id
#) t1 where t1.icustay_id in (select icustay_id from first_stay where los <=72)
#'''

query = '''
select t2.icustay_id, case when (t2.los > 4 or (extract(day from t2.dod - t2.first_intime) <= 4 and 
extract(day from t2.dod - t2.first_intime) >= 2) or (extract(day from t2.second_intime - t2.first_intime) <= 4 )) 
then 0 else 1 end as discharge from ( 
select pt.subject_id, pt.dod, t1.icustay_id, t1.first_intime, t1.first_outtime, t1.los, t1.second_intime, t1.second_outtime 
from patients pt join (
select first.subject_id, first.icustay_id, first.intime first_intime, first.outtime first_outtime, first.los los,
second.intime second_intime, second.outtime second_outtime from first_stay first left join (
select rk.subject_id, rk.icustay_id, rk.intime, rk.outtime from ( SELECT subject_id, icustay_id, intime,outtime, 
RANK() OVER (PARTITION BY subject_id ORDER BY intime asc) as RN
FROM icustays) rk where rk.rn=2) second on first.subject_id = second.subject_id where first.los>=2) 
t1 on pt.subject_id = t1.subject_id) t2
'''
labels = label_dict(pd.read_sql_query(query,conn))

In [30]:
len(labels)

23599

In [29]:
## select those who survive from first stay and only has one stay
#query = '''select stay.icustay_id, case when (dod is null or extract(day from dod - stay.outtime)>30) then 0 else 1 end as readmission 
#from first_stay stay join patients pt on stay.subject_id=pt.subject_id where stay.icustay_id in (
#select stay.icustay_id from first_stay stay join patients pt on stay.subject_id = pt.subject_id
#where pt.dod is null or pt.dod > stay.outtime and stay.subject_id in (select t1.subject_id from (select subject_id,
# count(icustay_id) stays from icustays group by subject_id) t1 where t1.stays=1)
#)
#'''
#labels1 = label_dict(pd.read_sql_query(query,conn))

In [23]:
# labels
# select subject_id given icustay_id
# select subject_id from icustays where icustay_id=12345 limit 1
# check label correctness
# select pt.subject_id, icu.icustay_id, pt.dod, icu.intime, icu.outtime from patients pt join icustays icu on pt.subject_id = icu.subject_id
# where pt.subject_id=123

## Merge time series, notes, and labels. Survivors from the first stay are considered.

In [10]:
data = {}

for stay_id in dynamic_features:
    if stay_id not in notes or stay_id not in labels:
        continue
    ts_times = sorted(list(dynamic_features[stay_id].keys()))
    note_times = sorted(list(notes[stay_id].keys()))
    start = ts_times[0]
    #end = max(ts_times[-1], note_times[-1])
    end = start + 48 * 3600
    ts_times = ts_times[:bisect.bisect_right(ts_times,end)]
    start_idx = bisect.bisect_left(note_times,start)
    end_idx = bisect.bisect_right(note_times,end)
    if start_idx == len(note_times):
        continue
    if start_idx == end_idx:
        if start_idx == 0:
            continue
        else:
            note_times = note_times[start_idx:end_idx+1]
    else:
        note_times = note_times[start_idx:end_idx]
    if len(ts_times) == 0 or len(note_times) == 0:
        continue
    data[stay_id] = {}
    data[stay_id]['dynamic'] = {}
    for timestamp in ts_times:
        data[stay_id]['dynamic'][timestamp] = dynamic_features[stay_id][timestamp]
    data[stay_id]['notes'] = {}
    for timestamp in note_times:
        data[stay_id]['notes'][timestamp] = notes[stay_id][timestamp]
    data[stay_id]['label'] = labels[stay_id]
    
path = './los_data.pickle'
file = open(path,'wb')
pickle.dump(data,file)
file.close()
    

In [11]:
len(data)

11472